# Run mlflow.genai.evaluate against the RAG endpoint with three custom Scorers, then run an SME calibration loop that reduces rater disagreement

The endpoint is live. The team's instincts say it works. The team's instincts are not a metric. You have one session to ship the first real eval: three Scorers, 20 rows, a hard inter-rater number. Round 1 will look bad; that is the point. Round 2 with a calibrated rubric will look better. The improvement is the takeaway, not the absolute number.

The hands-on work:

- Load a 20-row evaluation set fixture into a Delta table (question + ground_truth_answer + retrieved_chunk_ids).
- Define three custom `@mlflow.genai.scorer` Scorers: `correctness` (uses ground truth), `completeness` (reference-free), `helpfulness` (reference-free).
- Run `mlflow.genai.evaluate(data=..., predict_fn=<query Lab 9 endpoint>, scorers=[...])` and log the results to an MLflow experiment.
- Synthesize four SME rater scores on a sampled 10 rows (round 1), compute inter-rater agreement against the LLM judge scores, and persist the result. By design, round 1 agreement will be low.
- Apply the lab's calibration-effect synthesizer to produce round 2 SME ratings, re-compute agreement, and prove it improved past 0.6.

Estimated time: 70 minutes. Cost: $0.05 to $0.20 (judge LLM calls dominate the spend; 160 total FM API calls across two rounds).


In [ ]:
# NBVAL_SKIP
# Pinned dependencies. scikit-learn is added on top of the cross-cert pin
# because the inter-rater agreement metric uses sklearn's cohen_kappa_score.

!pip install --quiet labrun-checks==0.7.0 databricks-sdk==0.40.0 mlflow==2.20.0 langchain==0.3.7 databricks-langchain==0.1.1 scikit-learn==1.4.2


In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. UC identifiers (schema, table, volume) use
# snake_case under the labrun_ prefix because Unity Catalog identifiers prefer
# snake_case (hyphens force backtick-quoting everywhere downstream).

import atexit
import getpass
import json
import re
import sys
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "mlflow-genai-evaluate-with-llm-judges-and-sme-calibration"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID  # must equal cert YAML labs[].slug exactly

CATALOG = "workspace"
PARENT_SCHEMA = "default"
LAB_SCHEMA = "labrun_genai_eval"
LAB_SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{LAB_SCHEMA}"

EVAL_SET_TABLE_FQN = f"{LAB_SCHEMA_FQN}.eval_set"
SME_ROUND1_TABLE_FQN = f"{LAB_SCHEMA_FQN}.sme_round1"
SME_ROUND2_TABLE_FQN = f"{LAB_SCHEMA_FQN}.sme_round2"
AGREEMENT_TABLE_FQN = f"{LAB_SCHEMA_FQN}.agreement_scores"
SERVING_ENDPOINT_NAME = "labrun-rag-endpoint"
JUDGE_LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
MLFLOW_EXPERIMENT_NAME = None  # resolved post-register() once CALLER_USER is known
RANDOM_SEED = 42
SAMPLE_ROW_COUNT = 10
SME_COUNT = 4


In [ ]:
# NBVAL_SKIP
# Register the session, validate Databricks credentials via the SDK, and
# resolve the Starter SQL warehouse. This cell must succeed before the
# manifest cell runs anything.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass("Databricks workspace URL (https://...azuredatabricks.net or .cloud.databricks.com): ").strip()
databricks_token = getpass.getpass("Databricks personal access token (PAT): ").strip()

if not databricks_host or not databricks_token:
    print("Workspace URL and PAT are both required. Re-run this cell.")
    raise SystemExit(1)
if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"

w = WorkspaceClient(host=databricks_host, token=databricks_token)

try:
    me = w.current_user.me()
except Exception as exc:
    print("Databricks credentials are missing or expired. CurrentUser.me() failed:")
    print(f"  {exc}")
    print()
    print("Refresh your workspace URL and PAT, then re-run this cell.")
    raise SystemExit(1)

CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
print(f"Credentials validated. Workspace user: {CALLER_USER}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found in this workspace. Free Edition ships with")
    print("a Starter warehouse by default; if it has been deleted, recreate it")
    print("from the SQL > Warehouses page before re-running this cell.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
print(f"SQL warehouse resolved: {WAREHOUSE.name} ({WAREHOUSE_ID})")

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
}


def run_sql(statement, warehouse_id=None, wait_seconds=180):
    """Submit SQL to the warehouse and return {columns, rows}."""
    wid = warehouse_id or WAREHOUSE_ID
    resp = w.statement_execution.execute_statement(
        warehouse_id=wid, statement=statement, wait_timeout="50s"
    )
    statement_id = resp.statement_id
    deadline = time.time() + wait_seconds
    while True:
        state = resp.status.state if resp.status else None
        if state == StatementState.SUCCEEDED:
            break
        if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
            err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
            raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement}")
        if time.time() > deadline:
            raise TimeoutError(
                f"SQL did not finish in {wait_seconds}s. Last state: {state}. "
                f"The Starter warehouse may still be waking up; re-run the cell."
            )
        time.sleep(2)
        resp = w.statement_execution.get_statement(statement_id)
    columns = []
    if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
        columns = [c.name for c in resp.manifest.schema.columns]
    rows = []
    if resp.result and resp.result.data_array:
        rows = list(resp.result.data_array)
    return {"columns": columns, "rows": rows}


register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")


In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# Manifest is module-level and in reverse-creation order. Per
# RESOURCE-SAFETY-SPEC Rule 4 the orphan scan blocks execution if any
# tagged resources from a prior session exist.

USER_EMAIL = CALLER_USER

CLEANUP_MANIFEST = [
    CleanupResource(
        type="mlflow_experiment",
        id=f"/Users/{CALLER_USER}/labrun-genai-eval",
        region="databricks",
        cli_delete_command=(
            "databricks experiments delete-experiment "
            "$(databricks experiments get-by-name "
            f"--experiment-name /Users/{CALLER_USER}/labrun-genai-eval "
            "--output JSON | jq -r .experiment.experiment_id)"
        ),
    ),
    CleanupResource(
        type="uc_managed_table",
        id=AGREEMENT_TABLE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {AGREEMENT_TABLE_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=SME_ROUND2_TABLE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {SME_ROUND2_TABLE_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=SME_ROUND1_TABLE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {SME_ROUND1_TABLE_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=EVAL_SET_TABLE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {EVAL_SET_TABLE_FQN}\"",
    ),
    CleanupResource(
        type="uc_schema",
        id=LAB_SCHEMA_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP SCHEMA IF EXISTS {LAB_SCHEMA_FQN} CASCADE\"",
    ),
]
MLFLOW_EXPERIMENT_NAME = f"/Users/{CALLER_USER}/labrun-genai-eval"


class DatabricksCleanupAdapter:
    """Inline adapter for UC + MLflow + serving + vector search resources."""

    def delete_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "uc_managed_table":
            run_sql(f"DROP TABLE IF EXISTS {rid}")
        elif rtype == "uc_view":
            run_sql(f"DROP VIEW IF EXISTS {rid}")
        elif rtype == "uc_volume":
            run_sql(f"DROP VOLUME IF EXISTS {rid}")
        elif rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                listing = list(w.files.list_directory_contents(volume_path))
            except Exception:
                return
            for entry in listing:
                try:
                    if entry.is_directory:
                        w.files.delete_directory(entry.path)
                    else:
                        w.files.delete(entry.path)
                except Exception:
                    pass
        elif rtype == "uc_schema":
            run_sql(f"DROP SCHEMA IF EXISTS {rid} CASCADE")
        elif rtype == "uc_registered_model":
            try:
                import mlflow
                mlflow.set_registry_uri("databricks-uc")
                mlflow.MlflowClient().delete_registered_model(name=rid)
            except Exception:
                pass
        elif rtype == "mlflow_experiment":
            try:
                import mlflow
                exp = mlflow.get_experiment_by_name(rid)
                if exp is not None:
                    mlflow.delete_experiment(exp.experiment_id)
            except Exception:
                pass
        elif rtype == "model_serving_endpoint":
            try:
                w.serving_endpoints.delete(name=rid)
                deadline = time.time() + 600
                while time.time() < deadline:
                    try:
                        w.serving_endpoints.get(name=rid)
                    except Exception:
                        return
                    time.sleep(5)
            except Exception:
                pass
        elif rtype == "vector_search_index":
            try:
                w.vector_search_indexes.delete_index(index_name=rid)
            except Exception:
                pass
        elif rtype == "vector_search_endpoint":
            try:
                w.vector_search_endpoints.delete_endpoint(endpoint_name=rid)
            except Exception:
                pass
        else:
            raise ValueError(f"DatabricksCleanupAdapter: unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype in ("uc_managed_table", "uc_view"):
            parts = rid.split(".")
            if len(parts) != 3:
                return False
            catalog, schema, table = parts
            try:
                sql = (
                    "SELECT 1 FROM system.information_schema.tables "
                    f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                    f"AND table_name = '{table}'"
                )
                return len(run_sql(sql)["rows"]) > 0
            except Exception:
                return False
        if rtype == "uc_volume":
            parts = rid.split(".")
            if len(parts) != 3:
                return False
            catalog, schema, volume = parts
            try:
                sql = (
                    "SELECT 1 FROM system.information_schema.volumes "
                    f"WHERE volume_catalog = '{catalog}' AND volume_schema = '{schema}' "
                    f"AND volume_name = '{volume}'"
                )
                return len(run_sql(sql)["rows"]) > 0
            except Exception:
                return False
        if rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                listing = list(w.files.list_directory_contents(volume_path))
                return len(listing) > 0
            except Exception:
                return False
        if rtype == "uc_schema":
            parts = rid.split(".")
            if len(parts) != 2:
                return False
            catalog, schema = parts
            try:
                sql = (
                    "SELECT 1 FROM system.information_schema.schemata "
                    f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
                )
                return len(run_sql(sql)["rows"]) > 0
            except Exception:
                return False
        if rtype == "uc_registered_model":
            try:
                import mlflow
                mlflow.set_registry_uri("databricks-uc")
                mlflow.MlflowClient().get_registered_model(name=rid)
                return True
            except Exception:
                return False
        if rtype == "mlflow_experiment":
            try:
                import mlflow
                return mlflow.get_experiment_by_name(rid) is not None
            except Exception:
                return False
        if rtype == "model_serving_endpoint":
            try:
                w.serving_endpoints.get(name=rid)
                return True
            except Exception:
                return False
        if rtype == "vector_search_index":
            try:
                w.vector_search_indexes.get_index(index_name=rid)
                return True
            except Exception:
                return False
        if rtype == "vector_search_endpoint":
            try:
                w.vector_search_endpoints.get_endpoint(endpoint_name=rid)
                return True
            except Exception:
                return False
        return False

    def tag_scan(self, credentials, lab_slug, region):
        arns = []
        queries = [
            (
                "SELECT catalog_name, schema_name FROM system.information_schema.schema_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}",
            ),
            (
                "SELECT catalog_name, schema_name, table_name FROM system.information_schema.table_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
        ]
        for sql, fmt in queries:
            try:
                result = run_sql(sql)
            except Exception:
                continue
            for row in result["rows"]:
                arns.append(fmt(row))
        return arns


CLEANUP_ADAPTER = DatabricksCleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    return CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing UC objects tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom first, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")


## Resolve the prerequisite: the Lab 9 serving endpoint

This lab evaluates the `labrun-rag-endpoint` deployed in Lab 9 as the system under test. If that endpoint is missing (Lab 9 was never run, or Lab 9's cleanup ran), this cell halts with a clear message. Subsequent cells will skip rather than burn judge tokens against a non-existent endpoint.


In [ ]:
# NBVAL_SKIP
# Confirm the Lab 9 endpoint exists and is queryable before any judge calls.

try:
    ep = w.serving_endpoints.get(name=SERVING_ENDPOINT_NAME)
except Exception as exc:
    print(
        f"Prerequisite missing: serving endpoint {SERVING_ENDPOINT_NAME!r} "
        f"was not found. Run Lab 9 first to deploy it. Error: {exc!r}"
    )
    raise SystemExit(1)

ep_state = getattr(ep.state, "ready", None)
ep_state_value = getattr(ep_state, "value", ep_state) if ep_state is not None else None
if str(ep_state_value) != "READY":
    print(
        f"Endpoint {SERVING_ENDPOINT_NAME!r} is in state {ep_state_value!r}, "
        f"not READY. The first eval row would cold-start it; that is fine, but "
        f"if it is stuck, return to Lab 9 Task 1 and re-run the wait loop."
    )

print(f"Prerequisite endpoint resolved: {SERVING_ENDPOINT_NAME} (state={ep_state_value})")


## Stand up the schema, MLflow experiment, and registry wiring

The lab uses a fresh UC schema for the eval artifacts so cleanup is a single schema drop. The MLflow experiment lives under the per-user `/Users/<user>/labrun-genai-eval` path which Free Edition allows by default. The registry URI is set to `databricks-uc` even though this lab does not register a model; some `mlflow.genai` helpers consult the registry URI when resolving judge models.


In [ ]:
# NBVAL_SKIP
# Schema, experiment, and registry wiring.

import mlflow
import pandas as pd

run_sql(f"CREATE SCHEMA IF NOT EXISTS {LAB_SCHEMA_FQN}")
run_sql(f"ALTER SCHEMA {LAB_SCHEMA_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
print(f"MLflow experiment ready: {MLFLOW_EXPERIMENT_NAME}")


## Load the 20-row eval set fixture

The eval set is a fixed list of 20 questions with ground-truth answers and the chunk IDs the retriever should produce. The lab ships it inline so the eval is reproducible. In production the eval set lives in a Delta table maintained by SMEs and grows over time; the shape (question + ground_truth_answer + retrieved_chunk_ids) is the canonical one the exam guide names.


In [ ]:
# NBVAL_SKIP
# Eval set fixture: 20 questions tied to the Lab 9 RAG corpus.

EVAL_SET = [
    ("How do I sign up for Databricks Free Edition?",
     "Sign up at databricks.com via email OTP, Google, or Microsoft.",
     ["doc-1"]),
    ("What is Mosaic AI Vector Search Delta-Sync?",
     "A managed index that auto-updates from a source Delta table.",
     ["doc-2"]),
    ("How does the Foundation Model API bill?",
     "Pay-per-token endpoints bill cents per session for learning workloads.",
     ["doc-3"]),
    ("Explain three-level naming in Unity Catalog.",
     "Unity Catalog uses three levels: catalog.schema.table.",
     ["doc-4"]),
    ("Where do I find the Free Edition signup page?",
     "At databricks.com via email OTP, Google, or Microsoft sign-in.",
     ["doc-1"]),
    ("Do Delta-Sync indexes auto-refresh?",
     "Yes; they auto-update when the source Delta table changes.",
     ["doc-2"]),
    ("How much does the pay-per-token FM API cost per session?",
     "Cents per session for typical learning workloads.",
     ["doc-3"]),
    ("What are the three levels in a UC name?",
     "catalog, schema, and table or model.",
     ["doc-4"]),
    ("What sign-in options does Free Edition support?",
     "Email OTP, Google, and Microsoft.",
     ["doc-1"]),
    ("What triggers a Delta-Sync index refresh?",
     "Changes to the source Delta table.",
     ["doc-2"]),
    ("Which FM API tier is the cheapest for learning?",
     "Pay-per-token endpoints; cents per session.",
     ["doc-3"]),
    ("What naming convention does Unity Catalog enforce?",
     "Three-level naming: catalog.schema.table.",
     ["doc-4"]),
    ("Is Free Edition limited to email signup?",
     "No, it also supports Google and Microsoft sign-in.",
     ["doc-1"]),
    ("Does Vector Search require manual reindexing?",
     "No; Delta-Sync indexes auto-update from the source table.",
     ["doc-2"]),
    ("How are FM API tokens billed?",
     "Pay-per-token; cents per session for typical workloads.",
     ["doc-3"]),
    ("What is the third level in a UC name?",
     "The table or model name (after catalog and schema).",
     ["doc-4"]),
    ("Can I sign in with Microsoft?",
     "Yes, Microsoft sign-in is supported by Free Edition.",
     ["doc-1"]),
    ("What does the auto-update on Vector Search depend on?",
     "Changes detected on the source Delta table.",
     ["doc-2"]),
    ("What is the cost shape of pay-per-token endpoints?",
     "Cents per session for learning workloads.",
     ["doc-3"]),
    ("Give an example of a three-level UC name.",
     "workspace.default.my_table is a valid three-level UC name.",
     ["doc-4"]),
]

assert len(EVAL_SET) == 20, f"Eval set must have 20 rows, found {len(EVAL_SET)}"

eval_df = pd.DataFrame(
    EVAL_SET,
    columns=["question", "ground_truth_answer", "retrieved_chunk_ids"],
)

# Persist the eval set to Delta so the validator and any later replay can read it.
run_sql(
    f"CREATE OR REPLACE TABLE {EVAL_SET_TABLE_FQN} ("
    f"row_id INT, question STRING, ground_truth_answer STRING, retrieved_chunk_ids ARRAY<STRING>)"
)
run_sql(f"ALTER TABLE {EVAL_SET_TABLE_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")
for i, (q, gt, chunks) in enumerate(EVAL_SET):
    safe_q = q.replace(chr(39), chr(39) * 2)
    safe_gt = gt.replace(chr(39), chr(39) * 2)
    chunks_sql = ", ".join(f"'{c}'" for c in chunks)
    run_sql(
        f"INSERT INTO {EVAL_SET_TABLE_FQN} VALUES "
        f"({i}, '{safe_q}', '{safe_gt}', array({chunks_sql}))"
    )

print(f"Eval set persisted to {EVAL_SET_TABLE_FQN} ({len(EVAL_SET)} rows).")


## Define the predict function that hits the Lab 9 endpoint

`mlflow.genai.evaluate` takes a `predict_fn` callable that maps a pandas DataFrame of inputs to a list of predictions. The Lab 9 endpoint accepts `inputs=[<query>]` and returns predictions at `resp.predictions[0]`.

The function below catches per-row failures (cold-start blips, transient endpoint errors) and returns an explicit error string so the eval run completes with a per-row score column rather than crashing the whole run. The judge LLM then sees the error string in the response and scores it low on every dimension, which is what an outage should look like in the eval surface.


In [ ]:
# NBVAL_SKIP
# query_endpoint_fn: pandas in, list of predictions out, per-row error tolerance.

def query_endpoint_fn(df):
    questions = list(df["question"]) if "question" in df.columns else list(df.iloc[:, 0])
    predictions = []
    for q in questions:
        try:
            resp = w.serving_endpoints.query(name=SERVING_ENDPOINT_NAME, inputs=[q])
            text = resp.predictions[0]
            if not isinstance(text, str):
                text = str(text)
        except Exception as exc:
            text = f"[endpoint error: {exc!r}]"
        predictions.append(text)
    return predictions

print("query_endpoint_fn defined; wired to the Lab 9 endpoint.")


## Task 1: Define three custom Scorers

The `@mlflow.genai.scorer` decorator registers a function as a Scorer that `mlflow.genai.evaluate` can call per row. Three patterns matter for the exam:

- **Correctness** (ground-truth Scorer): the function signature accepts `request`, `response`, and `ground_truth`. The judge LLM compares the response to the ground truth and returns a 1 to 5 score plus a rationale.
- **Completeness** (reference-free): signature accepts `request` and `response` only. The judge LLM scores whether the response addresses every aspect of the question.
- **Helpfulness** (reference-free): signature accepts `request` and `response`. The judge scores actionability.

The validator checks: (1) all three Scorers exist and are callable, (2) `correctness` has `ground_truth` in its signature, (3) `completeness` and `helpfulness` do not.

The judge LLM is called via the chat completions API. The rubric strings stay short because the judge has to read them on every row; long rubrics inflate token spend with no signal gain.


In [ ]:
# NBVAL_SKIP
# Task 1: three custom Scorers via the @mlflow.genai.scorer decorator.

from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

# Fallback shim: some MLflow versions expose @scorer at mlflow.genai.scorers.scorer.
# Others expose it at mlflow.genai.scorer. The decorator below resolves whichever
# attribute exists at runtime so the lab works across the supported version window.
try:
    from mlflow.genai.scorers import scorer
except ImportError:
    from mlflow.genai import scorer  # type: ignore


def _judge(rubric, question, response, ground_truth=None):
    """Call the judge LLM and parse a 1-5 integer score from the first line."""
    gt_block = f"\nGround truth: {ground_truth}" if ground_truth is not None else ""
    user_msg = (
        f"Rubric: {rubric}\nQuestion: {question}\nResponse: {response}{gt_block}\n"
        "Reply with a single integer 1, 2, 3, 4, or 5 on the first line."
    )
    resp = w.serving_endpoints.query(
        name=JUDGE_LLM_ENDPOINT,
        messages=[
            ChatMessage(role=ChatMessageRole.SYSTEM,
                        content="You are a strict evaluator. Reply with one digit only."),
            ChatMessage(role=ChatMessageRole.USER, content=user_msg),
        ],
        max_tokens=8,
    )
    raw = (resp.choices[0].message.content or "").strip()
    for ch in raw:
        if ch in "12345":
            return int(ch)
    return 3  # safe default if the judge returned nothing parseable


# YOUR CODE: define @scorer-decorated `correctness(request, response, ground_truth)`
# YOUR CODE:   - rubric: 'Does the response match the ground truth answer?'
# YOUR CODE:   - return _judge(rubric, request, response, ground_truth=ground_truth)

# YOUR CODE: define @scorer-decorated `completeness(request, response)`
# YOUR CODE:   - rubric: 'Does the response address every aspect of the question?'
# YOUR CODE:   - return _judge(rubric, request, response)

# YOUR CODE: define @scorer-decorated `helpfulness(request, response)`
# YOUR CODE:   - rubric: 'Is the response actionable and useful for the asker?'
# YOUR CODE:   - return _judge(rubric, request, response)


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: three Scorers exist, callable, with the right signatures.

import inspect


def checkpoint_1(session):
    missing = [n for n in ("correctness", "completeness", "helpfulness") if n not in globals()]
    if missing:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Missing Scorers: {missing}. Define each as a @scorer-decorated function "
                f"per the Task 1 markdown."
            ),
        )
    for name in ("correctness", "completeness", "helpfulness"):
        fn = globals()[name]
        if not callable(fn):
            return CheckpointResult(
                status="fail",
                error_reason=f"Scorer {name!r} exists but is not callable: {fn!r}",
            )
    # Inspect signatures: correctness must accept ground_truth; the other two must not.
    def _params(fn):
        target = fn
        if hasattr(fn, "__wrapped__"):
            target = fn.__wrapped__
        try:
            return set(inspect.signature(target).parameters.keys())
        except (TypeError, ValueError):
            return set()
    correctness_params = _params(globals()["correctness"])
    completeness_params = _params(globals()["completeness"])
    helpfulness_params = _params(globals()["helpfulness"])
    if "ground_truth" not in correctness_params:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"correctness Scorer signature must accept 'ground_truth'; "
                f"observed parameters: {sorted(correctness_params)}."
            ),
        )
    if "ground_truth" in completeness_params:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"completeness Scorer must be reference-free (no 'ground_truth' "
                f"parameter); observed: {sorted(completeness_params)}."
            ),
        )
    if "ground_truth" in helpfulness_params:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"helpfulness Scorer must be reference-free (no 'ground_truth' "
                f"parameter); observed: {sorted(helpfulness_params)}."
            ),
        )
    return CheckpointResult(status="pass")


check(1, checkpoint_1)


<details><summary>Hint 1 (nudge)</summary>

Three `@scorer` definitions, each one a thin wrapper over `_judge(...)`. The differentiator across them is the rubric string and whether `ground_truth` appears in the signature.

</details>

<details><summary>Hint 2 (direction)</summary>

The shape per Scorer is roughly:

```
@scorer
def correctness(request, response, ground_truth):
    return _judge("rubric here", request, response, ground_truth=ground_truth)
```

Repeat for `completeness` and `helpfulness` but drop `ground_truth` from the signature and the `_judge` call.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
@scorer
def correctness(request, response, ground_truth):
    return _judge(
        "Does the response match the ground truth answer? 1 = wrong, 5 = exact match.",
        request, response, ground_truth=ground_truth,
    )


@scorer
def completeness(request, response):
    return _judge(
        "Does the response address every aspect of the question? 1 = misses most, 5 = covers all.",
        request, response,
    )


@scorer
def helpfulness(request, response):
    return _judge(
        "Is the response actionable and useful for the asker? 1 = useless, 5 = ready to act on.",
        request, response,
    )
```

</details>


**Wallet check.** Defining Scorers is free; the judge LLM has not been called yet. The first invoice line lands in Task 2 when `mlflow.genai.evaluate` triggers 60 judge calls (20 rows times 3 Scorers).


## Task 2: Run mlflow.genai.evaluate against the eval set

`mlflow.genai.evaluate(data=<DataFrame>, predict_fn=<callable>, scorers=[...])` walks each row, calls `predict_fn` to get the response, then calls every Scorer against the (request, response, ground_truth) triple. Inside an `mlflow.start_run()` the per-row scores plus aggregate `<scorer>/mean` metrics land on the run.

This single cell consumes 80 FM API calls: 20 endpoint queries plus 60 judge calls. If the Lab 9 endpoint scaled to zero, the first row eats a 30 to 90 second cold-start; subsequent rows are fast.

If the version of `mlflow.genai.evaluate` in your environment uses a different argument name (the API stabilized across MLflow 2.16 to 2.20), the lab's fallback wrapper at the bottom of the cell runs the equivalent loop by hand and logs the metrics manually so the rest of the lab proceeds.


In [ ]:
# NBVAL_SKIP
# Task 2: mlflow.genai.evaluate over the 20-row eval set.

import mlflow

# Capture the run_id and the per-row score frame for downstream tasks.
EVAL_RUN_ID = None
PER_ROW_SCORES = None  # pd.DataFrame with columns: row_id, response, correctness, completeness, helpfulness


def _manual_eval_fallback():
    """Hand-rolled equivalent of mlflow.genai.evaluate so the lab proceeds across
    MLflow version drift. Logs the same metric names a real eval would log."""
    responses = query_endpoint_fn(eval_df)
    rows = []
    for i, (row, response) in enumerate(zip(EVAL_SET, responses)):
        question, gt, _ = row
        c = correctness(request=question, response=response, ground_truth=gt)
        co = completeness(request=question, response=response)
        h = helpfulness(request=question, response=response)
        rows.append({
            "row_id": i, "question": question, "ground_truth": gt,
            "response": response,
            "correctness": c, "completeness": co, "helpfulness": h,
        })
    frame = pd.DataFrame(rows)
    mlflow.log_metric("correctness/mean", float(frame["correctness"].mean()))
    mlflow.log_metric("completeness/mean", float(frame["completeness"].mean()))
    mlflow.log_metric("helpfulness/mean", float(frame["helpfulness"].mean()))
    return frame


# YOUR CODE: open `with mlflow.start_run() as run:` and capture run.info.run_id into EVAL_RUN_ID
# YOUR CODE: try the canonical path:
#     mlflow.genai.evaluate(
#         data=eval_df,
#         predict_fn=query_endpoint_fn,
#         scorers=[correctness, completeness, helpfulness],
#     )
# YOUR CODE: in an except block, call _manual_eval_fallback() and assign the result to PER_ROW_SCORES.
# YOUR CODE:   On the canonical path, after the evaluate call returns, set PER_ROW_SCORES
# YOUR CODE:   to its `result.tables['eval_results_table']` DataFrame (or the equivalent
# YOUR CODE:   attribute on your MLflow version) AND make sure correctness/mean,
# YOUR CODE:   completeness/mean, helpfulness/mean metrics land on the run.


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: an MLflow run exists with the three /mean metrics and 20-row
# per-row results captured.


def checkpoint_2(session):
    import mlflow
    exp = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
    if exp is None:
        return CheckpointResult(
            status="fail",
            error_reason=f"MLflow experiment {MLFLOW_EXPERIMENT_NAME} not found.",
        )
    runs = mlflow.search_runs(
        experiment_ids=[exp.experiment_id],
        max_results=10,
        order_by=["attributes.start_time DESC"],
        output_format="list",
    )
    if not runs:
        return CheckpointResult(
            status="fail",
            error_reason="No MLflow runs found. Did mlflow.genai.evaluate run inside a start_run()?",
        )
    required = {"correctness/mean", "completeness/mean", "helpfulness/mean"}
    for run in runs:
        metrics = run.data.metrics if run.data else {}
        if required.issubset(set(metrics.keys())):
            # Also confirm the per-row frame in the notebook has 20 rows.
            frame = PER_ROW_SCORES
            if frame is None or len(frame) != 20:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"PER_ROW_SCORES has {0 if frame is None else len(frame)} rows; "
                        f"expected 20 (one per eval question). Capture eval_results into PER_ROW_SCORES."
                    ),
                )
            for col in ("correctness", "completeness", "helpfulness"):
                if col not in frame.columns:
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"PER_ROW_SCORES is missing column {col!r}; "
                            f"observed columns: {list(frame.columns)}"
                        ),
                    )
            return CheckpointResult(status="pass")
    return CheckpointResult(
        status="fail",
        error_reason=(
            "No recent run logged correctness/mean, completeness/mean, helpfulness/mean. "
            "Re-run mlflow.genai.evaluate inside a start_run() and confirm the metrics land."
        ),
    )


check(2, checkpoint_2)


<details><summary>Hint 1 (nudge)</summary>

Wrap the `mlflow.genai.evaluate` call inside `with mlflow.start_run() as run:`. The function returns an object whose `tables['eval_results_table']` holds the per-row scores; assign that to `PER_ROW_SCORES`. If your MLflow version raises on the call, fall through to `_manual_eval_fallback()` and assign its return value to `PER_ROW_SCORES`.

</details>

<details><summary>Hint 2 (direction)</summary>

```
with mlflow.start_run() as run:
    EVAL_RUN_ID = run.info.run_id
    try:
        result = mlflow.genai.evaluate(
            data=eval_df,
            predict_fn=query_endpoint_fn,
            scorers=[correctness, completeness, helpfulness],
        )
        PER_ROW_SCORES = result.tables["eval_results_table"]
    except Exception as exc:
        print(f"genai.evaluate raised ({exc!r}); using manual fallback.")
        PER_ROW_SCORES = _manual_eval_fallback()
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
with mlflow.start_run() as run:
    EVAL_RUN_ID = run.info.run_id
    try:
        result = mlflow.genai.evaluate(
            data=eval_df,
            predict_fn=query_endpoint_fn,
            scorers=[correctness, completeness, helpfulness],
        )
        tables = getattr(result, "tables", {}) or {}
        if "eval_results_table" in tables:
            PER_ROW_SCORES = tables["eval_results_table"]
        elif tables:
            PER_ROW_SCORES = next(iter(tables.values()))
        else:
            PER_ROW_SCORES = _manual_eval_fallback()
    except Exception as exc:
        print(f"genai.evaluate raised ({exc!r}); using manual fallback.")
        PER_ROW_SCORES = _manual_eval_fallback()

print(f"Eval run {EVAL_RUN_ID} captured {len(PER_ROW_SCORES)} per-row scores.")
print(PER_ROW_SCORES[["correctness", "completeness", "helpfulness"]].describe())
```

</details>


**Wallet check.** 80 FM API calls just landed: 20 endpoint queries plus 60 judge calls. At about 500 tokens per judge call (rubric plus question plus response) and 300 tokens per endpoint call, total tokens about 36,000. Spend: under 2 cents. The cold-start on row 1 cost you 30 to 90 seconds of wall clock; tokens were unaffected.


## Task 3: Synthesize round-1 SME ratings and compute inter-rater agreement

Sample 10 of the 20 rows. For each sampled row, the lab synthesizes 4 SME ratings using a deterministic seeded process that produces high variance. Then compute pairwise Cohen's kappa across the 4 SMEs (averaged) and persist the result to the agreement table as `round=1`.

This step uses Cohen's kappa pairwise because kappa is a two-rater metric. Krippendorff's alpha handles four raters in one shot but requires the `krippendorff` pypi package which is not in the lab's pin set. The exam guide accepts either; the choice docstring in the spoiler explains when to use which.

The fixture seeded with `RANDOM_SEED=42` produces an average pairwise kappa around 0.2 to 0.4 on round 1. The checkpoint asserts the persisted value is under 0.6.


In [ ]:
# NBVAL_SKIP
# Task 3: synthesize round-1 SME ratings + compute average pairwise Cohen's kappa.

import random
import itertools
from sklearn.metrics import cohen_kappa_score

rng = random.Random(RANDOM_SEED)
sample_indices = sorted(rng.sample(range(len(EVAL_SET)), SAMPLE_ROW_COUNT))
print(f"Sampled rows: {sample_indices}")


def synthesize_sme_round1(seed):
    """Round 1: high-variance SME ratings. Each SME draws around the row's
    LLM-judge correctness score with a wide noise spread (sigma~1.5) and clips
    to 1..5. Deterministic under the seed so the validator can re-compute."""
    r = random.Random(seed)
    judge_scores = list(PER_ROW_SCORES["correctness"])
    ratings = {f"sme_{i+1}": [] for i in range(SME_COUNT)}
    for idx in sample_indices:
        center = float(judge_scores[idx])
        for i in range(SME_COUNT):
            noisy = center + r.gauss(0, 1.5)
            clipped = max(1, min(5, int(round(noisy))))
            ratings[f"sme_{i+1}"].append(clipped)
    return ratings


def avg_pairwise_kappa(ratings):
    """Average Cohen's kappa across every (SME_i, SME_j) pair. Treats ratings
    as nominal categories; the exam guide names kappa as the pairwise option
    and Krippendorff's alpha as the four-rater option. The lab uses kappa
    averaged across pairs because sklearn ships it and our pin set is small."""
    names = list(ratings.keys())
    pair_scores = []
    for a, b in itertools.combinations(names, 2):
        try:
            k = cohen_kappa_score(ratings[a], ratings[b])
        except Exception:
            k = 0.0
        if k != k:  # NaN guard when both raters are constant
            k = 0.0
        pair_scores.append(float(k))
    return sum(pair_scores) / len(pair_scores) if pair_scores else 0.0


# Create the SME and agreement tables up front (idempotent under CREATE OR REPLACE).
run_sql(
    f"CREATE OR REPLACE TABLE {SME_ROUND1_TABLE_FQN} ("
    f"row_id INT, sme_1 INT, sme_2 INT, sme_3 INT, sme_4 INT)"
)
run_sql(f"ALTER TABLE {SME_ROUND1_TABLE_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")
run_sql(
    f"CREATE OR REPLACE TABLE {AGREEMENT_TABLE_FQN} ("
    f"round INT, agreement_metric DOUBLE, metric_name STRING)"
)
run_sql(f"ALTER TABLE {AGREEMENT_TABLE_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

# YOUR CODE: call sme_round1 = synthesize_sme_round1(seed=RANDOM_SEED)
# YOUR CODE: INSERT each (row_id, sme_1, sme_2, sme_3, sme_4) tuple into
#            SME_ROUND1_TABLE_FQN; row_id is sample_indices[i]
# YOUR CODE: kappa_round1 = avg_pairwise_kappa(sme_round1)
# YOUR CODE: INSERT INTO AGREEMENT_TABLE_FQN VALUES (1, kappa_round1, 'avg_pairwise_cohens_kappa')
# YOUR CODE: print(f"Round 1 average pairwise kappa: {kappa_round1:.3f}") and confirm it is below 0.6.


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: agreement_scores has a round=1 row with metric under 0.6
# and a sensible value range; also confirm sme_round1 has 10 rows.


def checkpoint_3(session):
    try:
        rows = run_sql(
            f"SELECT round, agreement_metric, metric_name FROM {AGREEMENT_TABLE_FQN} "
            f"WHERE round = 1"
        )["rows"]
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=f"SELECT against {AGREEMENT_TABLE_FQN} failed: {exc!r}",
        )
    if not rows:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"No round=1 row in {AGREEMENT_TABLE_FQN}. Did the INSERT INTO run?"
            ),
        )
    _, agreement, _ = rows[0]
    try:
        agreement = float(agreement)
    except Exception:
        return CheckpointResult(
            status="fail",
            error_reason=f"round=1 agreement_metric is not numeric: {agreement!r}",
        )
    if agreement < -1.0 or agreement > 1.0:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"round=1 agreement_metric={agreement} is outside the valid kappa range "
                f"of -1.0 to 1.0; recompute via avg_pairwise_kappa(sme_round1)."
            ),
        )
    if agreement >= 0.6:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"round=1 agreement_metric={agreement:.3f} is at or above 0.6. The fixture"
                f" is seeded to produce poor agreement on round 1; if you see a high value"
                f" the SME synthesizer was bypassed or seeded differently."
            ),
        )
    try:
        sme_rows = run_sql(
            f"SELECT count(*) FROM {SME_ROUND1_TABLE_FQN}"
        )["rows"]
        if not sme_rows or int(sme_rows[0][0]) != SAMPLE_ROW_COUNT:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{SME_ROUND1_TABLE_FQN} has "
                    f"{0 if not sme_rows else sme_rows[0][0]} rows; expected {SAMPLE_ROW_COUNT}."
                ),
            )
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=f"SELECT against {SME_ROUND1_TABLE_FQN} failed: {exc!r}",
        )
    return CheckpointResult(status="pass")


check(3, checkpoint_3)


<details><summary>Hint 1 (nudge)</summary>

`synthesize_sme_round1` returns a dict like `{sme_1: [...], sme_2: [...], sme_3: [...], sme_4: [...]}` where each list has 10 integer scores. INSERT one row per sampled index into `sme_round1`. Then call `avg_pairwise_kappa(sme_round1)` for the scalar. Persist that scalar as a single row into `agreement_scores` with `round=1`.

</details>

<details><summary>Hint 2 (direction)</summary>

```
sme_round1 = synthesize_sme_round1(seed=RANDOM_SEED)
for i, row_id in enumerate(sample_indices):
    run_sql(
        f"INSERT INTO {SME_ROUND1_TABLE_FQN} VALUES "
        f"({row_id}, {sme_round1['sme_1'][i]}, {sme_round1['sme_2'][i]}, "
        f"{sme_round1['sme_3'][i]}, {sme_round1['sme_4'][i]})"
    )
kappa_round1 = avg_pairwise_kappa(sme_round1)
run_sql(
    f"INSERT INTO {AGREEMENT_TABLE_FQN} VALUES (1, {kappa_round1}, 'avg_pairwise_cohens_kappa')"
)
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
sme_round1 = synthesize_sme_round1(seed=RANDOM_SEED)

for i, row_id in enumerate(sample_indices):
    run_sql(
        f"INSERT INTO {SME_ROUND1_TABLE_FQN} VALUES ("
        f"{row_id}, "
        f"{sme_round1['sme_1'][i]}, "
        f"{sme_round1['sme_2'][i]}, "
        f"{sme_round1['sme_3'][i]}, "
        f"{sme_round1['sme_4'][i]})"
    )

kappa_round1 = avg_pairwise_kappa(sme_round1)
run_sql(
    f"INSERT INTO {AGREEMENT_TABLE_FQN} VALUES "
    f"(1, {kappa_round1}, 'avg_pairwise_cohens_kappa')"
)
print(f"Round 1 average pairwise kappa: {kappa_round1:.3f}")
assert kappa_round1 < 0.6, "Round 1 fixture should produce poor agreement."
```

</details>


**Wallet check.** Synthesis and kappa math are pure Python; no judge calls fired in this task. The 80 calls from Task 2 are still your only invoice. Wall clock for this cell: under five seconds.


## Task 4: Calibrate, re-score, prove agreement improves

In production this is a meeting: read the rubric, score three calibration examples together, discuss disagreements, agree on canonical scores, then re-rate the eval set. The lab cannot bring four humans onto a Free Edition session, so it ships a calibration-effect synthesizer that simulates the post-calibration distribution: same SMEs, same 10 rows, but with the noise sigma reduced from 1.5 to 0.4. The deterministic seed plus the tighter sigma produces an average pairwise kappa above 0.6.

The exam-relevant takeaway: the **workflow** is rubric + calibration examples + re-score, and the **metric** that proves it worked is the inter-rater agreement number going up. The synthesis here only stands in for the part of the loop a lab cannot run.

The checkpoint asserts the persisted round-2 value is above 0.6 AND that it is strictly greater than the round-1 value.


In [ ]:
# NBVAL_SKIP
# Task 4: define a rubric, apply the calibration-effect synthesizer, re-score, persist.

RUBRIC = [
    "1 = response is wrong or off-topic.",
    "2 = response is partially relevant but missing the key fact.",
    "3 = response includes the key fact but with errors or omissions.",
    "4 = response is correct and addresses the question with minor gaps.",
    "5 = response is correct, complete, and useful as a direct answer.",
]
print("Calibration rubric (read together by all four SMEs):")
for line in RUBRIC:
    print(f"  {line}")


def synthesize_sme_round2(seed):
    """Round 2: post-calibration. Same SMEs, same rows, tighter noise (sigma 0.4).
    Deterministic under the seed so the validator can re-compute."""
    r = random.Random(seed + 1)  # advance the stream so round 2 is a distinct draw
    judge_scores = list(PER_ROW_SCORES["correctness"])
    ratings = {f"sme_{i+1}": [] for i in range(SME_COUNT)}
    for idx in sample_indices:
        center = float(judge_scores[idx])
        for i in range(SME_COUNT):
            noisy = center + r.gauss(0, 0.4)
            clipped = max(1, min(5, int(round(noisy))))
            ratings[f"sme_{i+1}"].append(clipped)
    return ratings


run_sql(
    f"CREATE OR REPLACE TABLE {SME_ROUND2_TABLE_FQN} ("
    f"row_id INT, sme_1 INT, sme_2 INT, sme_3 INT, sme_4 INT)"
)
run_sql(f"ALTER TABLE {SME_ROUND2_TABLE_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

# YOUR CODE: sme_round2 = synthesize_sme_round2(seed=RANDOM_SEED)
# YOUR CODE: INSERT each row into SME_ROUND2_TABLE_FQN (row_id is sample_indices[i])
# YOUR CODE: kappa_round2 = avg_pairwise_kappa(sme_round2)
# YOUR CODE: INSERT INTO AGREEMENT_TABLE_FQN VALUES (2, kappa_round2, 'avg_pairwise_cohens_kappa')
# YOUR CODE: print round 1 vs round 2 side by side and confirm round 2 > 0.6.


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: round=2 row exists, agreement > 0.6, and round 2 > round 1.


def checkpoint_4(session):
    try:
        rows = run_sql(
            f"SELECT round, agreement_metric FROM {AGREEMENT_TABLE_FQN} "
            f"ORDER BY round ASC"
        )["rows"]
    except Exception as exc:
        return CheckpointResult(
            status="error",
            error_reason=f"SELECT against {AGREEMENT_TABLE_FQN} failed: {exc!r}",
        )
    by_round = {int(r[0]): float(r[1]) for r in rows}
    if 2 not in by_round:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"No round=2 row in {AGREEMENT_TABLE_FQN}. Did the round-2 INSERT run?"
            ),
        )
    r2 = by_round[2]
    if r2 <= 0.6:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"round=2 agreement_metric={r2:.3f} is at or below 0.6. The calibration"
                f" effect synthesizer is seeded to produce >0.6; re-run Task 4 and confirm"
                f" you called synthesize_sme_round2(seed=RANDOM_SEED)."
            ),
        )
    if 1 in by_round and r2 <= by_round[1]:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"round=2 ({r2:.3f}) did not improve over round=1 ({by_round[1]:.3f}). "
                f"The calibration step is supposed to raise agreement; check the synthesizer call."
            ),
        )
    return CheckpointResult(status="pass")


check(4, checkpoint_4)


<details><summary>Hint 1 (nudge)</summary>

Same pattern as Task 3 with one substitution: call `synthesize_sme_round2` instead of `synthesize_sme_round1`, write to `sme_round2`, persist with `round=2`. The tighter sigma is what raises the kappa.

</details>

<details><summary>Hint 2 (direction)</summary>

```
sme_round2 = synthesize_sme_round2(seed=RANDOM_SEED)
for i, row_id in enumerate(sample_indices):
    run_sql(
        f"INSERT INTO {SME_ROUND2_TABLE_FQN} VALUES ("
        f"{row_id}, {sme_round2['sme_1'][i]}, {sme_round2['sme_2'][i]}, "
        f"{sme_round2['sme_3'][i]}, {sme_round2['sme_4'][i]})"
    )
kappa_round2 = avg_pairwise_kappa(sme_round2)
run_sql(
    f"INSERT INTO {AGREEMENT_TABLE_FQN} VALUES (2, {kappa_round2}, 'avg_pairwise_cohens_kappa')"
)
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4:

```python
sme_round2 = synthesize_sme_round2(seed=RANDOM_SEED)

for i, row_id in enumerate(sample_indices):
    run_sql(
        f"INSERT INTO {SME_ROUND2_TABLE_FQN} VALUES ("
        f"{row_id}, "
        f"{sme_round2['sme_1'][i]}, "
        f"{sme_round2['sme_2'][i]}, "
        f"{sme_round2['sme_3'][i]}, "
        f"{sme_round2['sme_4'][i]})"
    )

kappa_round2 = avg_pairwise_kappa(sme_round2)
run_sql(
    f"INSERT INTO {AGREEMENT_TABLE_FQN} VALUES "
    f"(2, {kappa_round2}, 'avg_pairwise_cohens_kappa')"
)
print(f"Round 1 kappa: {kappa_round1:.3f}")
print(f"Round 2 kappa: {kappa_round2:.3f}")
print(f"Delta: +{kappa_round2 - kappa_round1:.3f}")
assert kappa_round2 > 0.6, "Round 2 fixture should clear 0.6."
assert kappa_round2 > kappa_round1, "Calibration must raise kappa."
```

</details>


**Wallet check.** Still no new judge calls; the round-2 SME ratings are synthesized in Python and kappa is sklearn. The full session token tally is still around 36,000 from Task 2's evaluate run. Total spend rounds to under 5 cents.


## Cleanup

Tear down the MLflow experiment first, then the four Delta tables in reverse-creation order (agreement_scores, sme_round2, sme_round1, eval_set), then the lab schema. The Lab 9 serving endpoint stays up. If you are done with the cert track entirely, return to Lab 9 and run its cleanup cell to free the active-endpoint slot.


In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order using the inline Databricks adapter. Prints the canonical summary
# from RESOURCE-SAFETY-SPEC Rule 6 and sys.exit(1) on any failure.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold lab objects. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)


**Session total: about $0.05.** 80 FM API calls from the single evaluate run, fractions of a cent each, all logged on the MLflow run for later inspection. Calibration math and kappa computations were free. This is the priciest free lab in the cert. Your coffee still cost more.


## Reflection

These are not graded. They are for you.

1. Walk through what changed between round 1 (poor agreement) and round 2 (better agreement) in this lab. The change was synthetic, but the workflow is real: rubric + calibration examples + re-score. What is the production version of this step (the version where you cannot synthesize the SME scores)?

2. The three Scorers (correctness, completeness, helpfulness) measure different dimensions. If you had to pick ONE Scorer to monitor in production (one number, one alert threshold), which one would you pick and why? What use case would change your answer?

## Exam-Style Practice

**Question 1 (Domain 6, SME feedback incorporation per the official sample-question pattern):**

A GenAI engineer is evaluating a customer-support RAG assistant used internally by operations teams. Four domain experts review sampled answers each week in MLflow using dimensions such as factual accuracy, completeness, and usefulness. After several rounds, the engineer notices that expert ratings vary widely for the same responses, making the evaluation data unreliable for tracking model improvements over time. What should the engineer do?

A. Use an LLM-as-a-judge to rescore past and future responses, and treat the model-generated ratings as the primary source of truth instead of reconciling expert disagreement.
B. Define clear rubrics, calibrate SMEs on the criteria, and use the aligned judgments in `mlflow.genai.evaluate()` for consistent agent evaluation.
C. Average the scores from all domain experts for each response and use the blended score directly as the definitive benchmark for model tuning.
D. Build the benchmark only from responses where all experts already agree, and exclude disputed cases from the evaluation set to improve consistency.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: replacing humans with an LLM judge does not fix the rubric ambiguity; it just hides the disagreement under a single rater's bias. The exam guide explicitly names SME calibration as the canonical workflow.
- B is correct: the canonical fix for high inter-rater variance is a shared, written rubric plus a calibration step (read the rubric, score the same calibration examples together, discuss disagreement). The aligned judgments then feed `mlflow.genai.evaluate()` as the eval ground truth.
- C is wrong: averaging hides the disagreement; the blended score is precise but not accurate. Two SMEs scoring 1 and 5 average to 3, which is not what either thinks the answer is.
- D is wrong: filtering to the easy cases produces a biased eval set; the disputed cases are exactly where the model needs to improve.

This is verbatim the Question 10 pattern from the official sample-question set in the March 2026 exam guide.

</details>

**Question 2 (Domain 6, ground-truth vs reference-free judges):**

A GenAI engineer is defining Scorers for an MLflow GenAI evaluation. Which Scorer REQUIRES a ground-truth answer column in the eval set?

A. Toxicity (does the response contain toxic content?).
B. Faithfulness (does the response contradict any retrieved chunk?).
C. Correctness (does the response match the expected answer?).
D. Helpfulness (does the response give actionable next steps?).

<details><summary>Show answer</summary>

**Correct: C.**

- A is wrong: toxicity is reference-free; the judge scores the response text alone.
- B is wrong: faithfulness compares the response to retrieved CHUNKS (which the system retrieves at inference time), not to a separate ground-truth answer column.
- C is correct: correctness requires a known "right" answer to compare against. The eval set must include a `ground_truth_answer` column.
- D is wrong: helpfulness is reference-free; the judge evaluates whether the response is actionable independent of any ground truth.

Section 6 of the exam guide names "evaluation judges that require ground truth" as a distinct task statement.

</details>
